# USD Protective Hedge Dynamics

This notebook is an executable walkthrough of the current USD-first multi-asset traffic-light model and the protective ETH short overlay added in commit `d5ea91d`.

The goal is not to optimize. The goal is to understand the path mechanics before the next development stage:

- reproduce the current checkpoint and the new protective-hedge winner;
- compare final USD, SOL-equivalent value, max drawdown, post-2024 drawdown, Sortino, and health factor;
- inspect when the hedge was active and whether it helped or hurt recovery;
- drill into worst drawdown windows with collateral, debt, traffic-light, and risk state side by side.

Reference result from the refinement sweep:

| Scenario | Final USD | Final SOL | Max DD |
| --- | ---: | ---: | ---: |
| Anchor, no hedge | $66,475.12 | 532.014 | 59.294% |
| Light protective hedge | $68,291.65 | 546.552 | 57.440% |

In [ ]:
from __future__ import annotations

import math
import sys
from html import escape
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "arblab").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from arblab.backtest.data import OHLCVConfig, fetch_ohlcv
from arblab.backtest.engine import BacktestEngine, EngineConfig
from arblab.backtest.traffic_lights import asset_green_counts_by_bar
from arblab.strategies.multi_asset_traffic_light import MultiAssetTrafficLightStrategy

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## Configuration

The default configuration below reproduces the refinement report. Edit only the `CUSTOM_HEDGE_FLOORS` cell later if you want to test a new floor without changing strategy code.

In [ ]:
START = "2021-01-01"
END = "2025-12-31"
INITIAL_SOL = 100.0
SYMBOLS = ["SOL", "ETH"]
TIMEFRAMES = ("1h", "4h", "8h", "1d")
LOOKBACK_BARS = 24 * 14
FINAL_SOL_PRICE = None

ANCHOR_SCENARIO = {
    "name": "anchor_no_hedge",
    "enable_protective_short_hedge": False,
    "protective_hedge_floors": {},
}

BEST_HEDGE_SCENARIO = {
    "name": "light_hedge_g3_0.050_g2_0.025_r_0.000",
    "enable_protective_short_hedge": True,
    "protective_short_symbol": "ETH",
    "protective_hedge_floors": {4: 0.00, 3: 0.050, 2: 0.025, 1: 0.00, 0: 0.00},
}

# Optional editable scenario. Run the custom scenario cell near the bottom after changing this.
CUSTOM_HEDGE_FLOORS = {4: 0.00, 3: 0.050, 2: 0.025, 1: 0.00, 0: 0.00}

## Helper Functions

These helpers intentionally use only pandas and inline SVG/HTML so the notebook works in the current repo environment without adding plot libraries.

In [ ]:
def signal_tiers() -> list[dict[str, float]]:
    return [
        {"green": 3, "target_long_fraction": 1.025},
        {"green": 4, "target_long_fraction": 1.075},
    ]


def max_drawdown_pct(values: pd.Series) -> float:
    running_peak = values.cummax()
    drawdown = (values - running_peak) / running_peak
    return abs(float(drawdown.min())) * 100.0


def drawdown_series(values: pd.Series) -> pd.Series:
    running_peak = values.cummax()
    return (values / running_peak - 1.0) * 100.0


def drawdown_window(history: pd.DataFrame, start: str) -> float:
    window = history.loc[history.index >= pd.Timestamp(start, tz="UTC")]
    if window.empty:
        return 0.0
    return max_drawdown_pct(window["portfolio_value"])


def load_price_data() -> pd.DataFrame:
    return fetch_ohlcv(
        symbols=[
            OHLCVConfig(symbol="SOL/USDT", display_name="SOL"),
            OHLCVConfig(symbol="ETH/USDT", display_name="ETH"),
        ],
        timeframe="1h",
        start=START,
        end=END,
        use_cache=True,
    )


def build_config(price_data: pd.DataFrame, scenario: dict[str, object]) -> dict[str, object]:
    first = price_data.iloc[0]
    return {
        "initial_collateral_symbol": "SOL",
        "initial_collateral_amount": INITIAL_SOL,
        "directional_symbols": SYMBOLS,
        "initial_prices": {
            "SOL": float(first[("SOL", "close")]),
            "ETH": float(first[("ETH", "close")]),
        },
        "atr_period": 10,
        "supertrend_multiplier": 3.0,
        "timeframes": TIMEFRAMES,
        "min_long_green": 3,
        "max_short_green": -1,
        "target_long_fraction": 1.075,
        "target_short_fraction": 0.00,
        "enable_signal_governor": True,
        "signal_exposure_tiers": signal_tiers(),
        "rebalance_threshold": 0.05,
        "swap_fee_bps": 10.0,
        **scenario,
    }


def run_scenario(price_data: pd.DataFrame, green_scores: pd.DataFrame, scenario: dict[str, object]):
    config = build_config(price_data, scenario)
    config["green_scores"] = green_scores
    result = BacktestEngine(
        MultiAssetTrafficLightStrategy(),
        EngineConfig(lookback_bars=LOOKBACK_BARS),
    ).run(price_data, config)
    return result, config


def enrich_history(history: pd.DataFrame, price_data: pd.DataFrame) -> pd.DataFrame:
    out = history.copy()
    aligned_prices = price_data.reindex(out.index)
    out["SOL_close"] = aligned_prices[("SOL", "close")]
    out["ETH_close"] = aligned_prices[("ETH", "close")]
    out["sol_equiv"] = out["portfolio_value"] / out["SOL_close"]
    out["drawdown_pct"] = drawdown_series(out["portfolio_value"])
    out["sol_return_pct"] = (out["SOL_close"] / out["SOL_close"].iloc[0] - 1.0) * 100.0
    out["model_return_pct"] = (out["portfolio_value"] / out["portfolio_value"].iloc[0] - 1.0) * 100.0
    out["eth_short_value_pct_equity"] = out["debt_ETH_value"] / out["portfolio_value"].replace(0, pd.NA)
    out["long_collateral_value"] = out.get("collateral_SOL_value", 0.0) + out.get("collateral_ETH_value", 0.0)
    out["gross_exposure_value"] = out["long_collateral_value"] + out.get("debt_ETH_value", 0.0) + out.get("debt_USDC_value", 0.0)
    return out


def metric_row(name: str, result, history: pd.DataFrame, final_sol_price: float) -> dict[str, object]:
    return {
        "scenario": name,
        "final_usd": result.metrics.final_value,
        "final_sol_equiv": result.metrics.final_value / final_sol_price,
        "max_dd_pct": result.metrics.max_drawdown_pct,
        "post_2024_dd_pct": drawdown_window(history, "2024-01-01"),
        "sortino": result.metrics.sortino_ratio,
        "min_health_factor": result.metrics.min_health_factor,
        "bars_hf_below_1_5": result.metrics.bars_below_hf_1_5,
        "actions": result.metrics.total_actions,
        "interest_paid": result.metrics.total_interest_paid,
        "liquidations": result.metrics.total_liquidations,
    }


def fmt_money(x: float) -> str:
    return f"${x:,.2f}"


def fmt_pct(x: float) -> str:
    return f"{x:.3f}%"

In [ ]:
class HTMLBlock:
    def __init__(self, html: str):
        self.html = html

    def _repr_html_(self) -> str:
        return self.html

    def __repr__(self) -> str:
        return self.html


def show_html(html: str):
    try:
        from IPython.display import HTML, display
        display(HTML(html))
    except Exception:
        print(html)


def _nice_ticks(vmin: float, vmax: float, n: int = 4) -> list[float]:
    if not math.isfinite(vmin) or not math.isfinite(vmax) or vmin == vmax:
        return [vmin]
    return [vmin + (vmax - vmin) * i / max(n - 1, 1) for i in range(n)]


def svg_lines(
    series_map: dict[str, pd.Series],
    title: str,
    height: int = 280,
    width: int = 980,
    y_label: str = "",
    colors: tuple[str, ...] = ("#2563eb", "#dc2626", "#16a34a", "#9333ea", "#ea580c"),
) -> HTMLBlock:
    clean = {name: s.dropna().astype(float) for name, s in series_map.items() if not s.dropna().empty}
    if not clean:
        return HTMLBlock(f"<p><strong>{escape(title)}</strong>: no data</p>")

    all_values = pd.concat(clean.values())
    vmin = float(all_values.min())
    vmax = float(all_values.max())
    if vmin == vmax:
        vmin -= 1.0
        vmax += 1.0
    pad = (vmax - vmin) * 0.08
    vmin -= pad
    vmax += pad

    margin = {"left": 78, "right": 24, "top": 34, "bottom": 48}
    inner_w = width - margin["left"] - margin["right"]
    inner_h = height - margin["top"] - margin["bottom"]

    def x_for(i: int, n: int) -> float:
        return margin["left"] + (inner_w * i / max(n - 1, 1))

    def y_for(v: float) -> float:
        return margin["top"] + inner_h - ((v - vmin) / (vmax - vmin) * inner_h)

    grid = []
    for tick in _nice_ticks(vmin, vmax, 5):
        y = y_for(tick)
        grid.append(f'<line x1="{margin["left"]}" y1="{y:.2f}" x2="{width-margin["right"]}" y2="{y:.2f}" stroke="#e5e7eb"/>')
        grid.append(f'<text x="{margin["left"]-8}" y="{y+4:.2f}" text-anchor="end" font-size="11" fill="#475569">{tick:,.1f}</text>')

    paths = []
    legend = []
    for idx, (name, series) in enumerate(clean.items()):
        points = []
        values = series.reset_index(drop=True)
        for i, value in enumerate(values):
            points.append(f"{x_for(i, len(values)):.2f},{y_for(float(value)):.2f}")
        color = colors[idx % len(colors)]
        paths.append(f'<polyline fill="none" stroke="{color}" stroke-width="2" points="{" ".join(points)}"/>')
        lx = margin["left"] + idx * 190
        ly = height - 16
        legend.append(f'<line x1="{lx}" y1="{ly}" x2="{lx+24}" y2="{ly}" stroke="{color}" stroke-width="3"/>')
        legend.append(f'<text x="{lx+32}" y="{ly+4}" font-size="12" fill="#111827">{escape(name)}</text>')

    html = f"""
    <div style="margin: 12px 0 24px 0;">
      <svg viewBox="0 0 {width} {height}" width="100%" style="max-width:{width}px; border:1px solid #e5e7eb; background:white;">
        <text x="{margin['left']}" y="22" font-size="16" font-weight="600" fill="#111827">{escape(title)}</text>
        <text x="18" y="{height/2:.0f}" transform="rotate(-90 18 {height/2:.0f})" font-size="12" fill="#475569">{escape(y_label)}</text>
        {''.join(grid)}
        <line x1="{margin['left']}" y1="{height-margin['bottom']}" x2="{width-margin['right']}" y2="{height-margin['bottom']}" stroke="#94a3b8"/>
        <line x1="{margin['left']}" y1="{margin['top']}" x2="{margin['left']}" y2="{height-margin['bottom']}" stroke="#94a3b8"/>
        {''.join(paths)}
        {''.join(legend)}
      </svg>
    </div>
    """
    return HTMLBlock(html)


def format_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ["final_usd", "interest_paid"]:
        out[col] = out[col].map(lambda x: f"${x:,.2f}")
    for col in ["final_sol_equiv", "sortino", "min_health_factor"]:
        out[col] = out[col].map(lambda x: f"{x:,.3f}")
    for col in ["max_dd_pct", "post_2024_dd_pct"]:
        out[col] = out[col].map(lambda x: f"{x:,.3f}%")
    return out

## Load Data and Run the Two Key Scenarios

This cell fetches cached Binance OHLCV data for SOL and ETH, computes the traffic-light scores once, and runs the anchor plus the light hedge winner.

Expected runtime on cached data is roughly one minute.

In [ ]:
price_data = load_price_data()
green_scores = asset_green_counts_by_bar(
    price_data,
    symbols=SYMBOLS,
    atr_period=10,
    multiplier=3.0,
    timeframes=TIMEFRAMES,
)
FINAL_SOL_PRICE = float(price_data.iloc[-1][("SOL", "close")])

anchor_result, anchor_config = run_scenario(price_data, green_scores, ANCHOR_SCENARIO)
hedge_result, hedge_config = run_scenario(price_data, green_scores, BEST_HEDGE_SCENARIO)

anchor_history = enrich_history(anchor_result.history, price_data)
hedge_history = enrich_history(hedge_result.history, price_data)

metrics = pd.DataFrame([
    metric_row("anchor_no_hedge", anchor_result, anchor_history, FINAL_SOL_PRICE),
    metric_row("light_protective_hedge", hedge_result, hedge_history, FINAL_SOL_PRICE),
])
format_metrics(metrics)

## Headline Delta

This is the main decision checkpoint: the hedge has to reduce drawdown while preserving or improving final USD and SOL-equivalent value.

In [ ]:
delta = pd.DataFrame([
    {
        "metric": "Final USD",
        "anchor": anchor_result.metrics.final_value,
        "hedge": hedge_result.metrics.final_value,
        "delta": hedge_result.metrics.final_value - anchor_result.metrics.final_value,
    },
    {
        "metric": "Final SOL equiv",
        "anchor": anchor_result.metrics.final_value / FINAL_SOL_PRICE,
        "hedge": hedge_result.metrics.final_value / FINAL_SOL_PRICE,
        "delta": hedge_result.metrics.final_value / FINAL_SOL_PRICE - anchor_result.metrics.final_value / FINAL_SOL_PRICE,
    },
    {
        "metric": "Max DD pct",
        "anchor": anchor_result.metrics.max_drawdown_pct,
        "hedge": hedge_result.metrics.max_drawdown_pct,
        "delta": hedge_result.metrics.max_drawdown_pct - anchor_result.metrics.max_drawdown_pct,
    },
    {
        "metric": "Post-2024 DD pct",
        "anchor": drawdown_window(anchor_history, "2024-01-01"),
        "hedge": drawdown_window(hedge_history, "2024-01-01"),
        "delta": drawdown_window(hedge_history, "2024-01-01") - drawdown_window(anchor_history, "2024-01-01"),
    },
])

display_delta = delta.copy()
display_delta["anchor"] = display_delta.apply(lambda r: fmt_money(r.anchor) if r.metric == "Final USD" else f"{r.anchor:,.3f}", axis=1)
display_delta["hedge"] = display_delta.apply(lambda r: fmt_money(r.hedge) if r.metric == "Final USD" else f"{r.hedge:,.3f}", axis=1)
display_delta["delta"] = display_delta.apply(lambda r: fmt_money(r.delta) if r.metric == "Final USD" else f"{r.delta:,.3f}", axis=1)
display_delta

## Equity and Drawdown Path

The hedge result is not just lower max drawdown. It also changes the compounding path. Start here when trying to understand whether a change is robust or just a final-mark artifact.

In [ ]:
svg_lines(
    {
        "Anchor USD": anchor_history["portfolio_value"],
        "Light hedge USD": hedge_history["portfolio_value"],
    },
    "Portfolio Value, USD",
    y_label="USD",
)

In [ ]:
svg_lines(
    {
        "Anchor drawdown": anchor_history["drawdown_pct"],
        "Light hedge drawdown": hedge_history["drawdown_pct"],
    },
    "Drawdown from Running Peak",
    y_label="Drawdown %",
)

In [ ]:
svg_lines(
    {
        "Anchor SOL equiv": anchor_history["sol_equiv"],
        "Light hedge SOL equiv": hedge_history["sol_equiv"],
    },
    "Portfolio Value Expressed in SOL",
    y_label="SOL equiv",
)

## Traffic-Light and Exposure State

These plots show how the model changes risk as the four traffic lights evolve. The key hedge finding was that a very light hedge in green-3/green-2 states helped, while forcing hedge in red states hurt recovery.

In [ ]:
svg_lines(
    {
        "Anchor long target": anchor_history["target_long_fraction"],
        "Hedge long target": hedge_history["target_long_fraction"],
        "Hedge short target": hedge_history["target_short_fraction"],
    },
    "Target Long and Short Fractions",
    y_label="Fraction",
)

In [ ]:
svg_lines(
    {
        "SOL green count, anchor": anchor_history["long_green"],
        "SOL green count, hedge": hedge_history["long_green"],
    },
    "Selected Long Green Count",
    y_label="Green lights",
)

In [ ]:
svg_lines(
    {
        "Hedge ETH debt value": hedge_history["debt_ETH_value"],
        "Hedge USDC collateral": hedge_history["collateral_USDC_value"],
        "Hedge SOL collateral": hedge_history["collateral_SOL_value"],
    },
    "Hedge Scenario Position State",
    y_label="USD value",
)

In [ ]:
svg_lines(
    {
        "Anchor health factor": anchor_history["health_factor"].clip(upper=10),
        "Hedge health factor": hedge_history["health_factor"].clip(upper=10),
    },
    "Health Factor, Clipped at 10 for Readability",
    y_label="HF",
)

## Hedge Activation Summary

This table shows how often the hedge target was nonzero and what the account looked like when it was active.

In [ ]:
active = hedge_history[hedge_history["target_short_fraction"] > 0].copy()
activation_summary = pd.DataFrame([
    {"measure": "Total bars", "value": len(hedge_history)},
    {"measure": "Bars with target_short_fraction > 0", "value": len(active)},
    {"measure": "Active share of bars", "value": f"{len(active) / len(hedge_history) * 100:.2f}%"},
    {"measure": "Average target short fraction when active", "value": f"{active['target_short_fraction'].mean():.4f}" if not active.empty else "0.0000"},
    {"measure": "Max target short fraction", "value": f"{hedge_history['target_short_fraction'].max():.4f}"},
    {"measure": "Max ETH debt value", "value": fmt_money(float(hedge_history["debt_ETH_value"].max()))},
    {"measure": "Final ETH debt value", "value": fmt_money(float(hedge_history["debt_ETH_value"].iloc[-1]))},
])
activation_summary

In [ ]:
by_short_target = (
    hedge_history.assign(target_short_bucket=hedge_history["target_short_fraction"].round(3))
    .groupby("target_short_bucket")
    .agg(
        bars=("portfolio_value", "size"),
        mean_hourly_pnl=("portfolio_value", lambda s: s.diff().mean()),
        avg_drawdown_pct=("drawdown_pct", "mean"),
        min_drawdown_pct=("drawdown_pct", "min"),
        avg_eth_debt_value=("debt_ETH_value", "mean"),
        avg_health_factor=("health_factor", lambda s: s.clip(upper=10).mean()),
    )
    .reset_index()
)
by_short_target

## Worst Drawdown Window

The next cells identify the worst drawdown timestamp for each scenario and show the local state around those points. This is where we can see whether the hedge improved the actual stress window or merely shifted the path elsewhere.

In [ ]:
def worst_drawdown_time(history: pd.DataFrame) -> pd.Timestamp:
    return history["drawdown_pct"].idxmin()


def window_table(history: pd.DataFrame, center: pd.Timestamp, days: int = 21) -> pd.DataFrame:
    start = center - pd.Timedelta(days=days)
    end = center + pd.Timedelta(days=days)
    cols = [
        "portfolio_value",
        "sol_equiv",
        "drawdown_pct",
        "SOL_close",
        "ETH_close",
        "selected_long",
        "long_green",
        "target_long_fraction",
        "target_short_fraction",
        "collateral_SOL_value",
        "collateral_USDC_value",
        "debt_USDC_value",
        "debt_ETH_value",
        "health_factor",
        "action_count",
    ]
    sample = history.loc[start:end, cols].copy()
    return sample.resample("12h").last().dropna(how="all")

anchor_worst = worst_drawdown_time(anchor_history)
hedge_worst = worst_drawdown_time(hedge_history)

pd.DataFrame([
    {"scenario": "anchor_no_hedge", "worst_drawdown_time": anchor_worst, "worst_dd_pct": anchor_history.loc[anchor_worst, "drawdown_pct"]},
    {"scenario": "light_protective_hedge", "worst_drawdown_time": hedge_worst, "worst_dd_pct": hedge_history.loc[hedge_worst, "drawdown_pct"]},
])

In [ ]:
anchor_window = window_table(anchor_history, anchor_worst, days=21)
hedge_window = window_table(hedge_history, hedge_worst, days=21)

# Display a compact tail around the stress point. Use the full dataframes above for deeper inspection.
hedge_window.tail(18)

In [ ]:
svg_lines(
    {
        "Anchor DD around worst": anchor_history.loc[anchor_worst - pd.Timedelta(days=45): anchor_worst + pd.Timedelta(days=45), "drawdown_pct"],
        "Hedge DD around hedge worst": hedge_history.loc[hedge_worst - pd.Timedelta(days=45): hedge_worst + pd.Timedelta(days=45), "drawdown_pct"],
    },
    "Local Drawdown Around Worst Stress Windows",
    y_label="Drawdown %",
)

## Quarter-by-Quarter Decomposition

This is a compact play-by-play table. It helps answer whether the improvement came from one crisis window, better re-entry, less drag, or a changed final exposure.

In [ ]:
def quarterly_decomposition(history: pd.DataFrame) -> pd.DataFrame:
    q = history.resample("QE").agg(
        start_value=("portfolio_value", "first"),
        end_value=("portfolio_value", "last"),
        min_value=("portfolio_value", "min"),
        end_sol_equiv=("sol_equiv", "last"),
        avg_long_target=("target_long_fraction", "mean"),
        avg_short_target=("target_short_fraction", "mean"),
        max_eth_debt=("debt_ETH_value", "max"),
        min_health_factor=("health_factor", "min"),
        actions=("action_count", "sum"),
    )
    q["quarter_return_pct"] = (q["end_value"] / q["start_value"] - 1.0) * 100.0
    q["quarter_dd_pct"] = q.apply(lambda r: (r["min_value"] / r["start_value"] - 1.0) * 100.0, axis=1)
    q.index = [f"{ts.year}Q{((ts.month - 1) // 3) + 1}" for ts in q.index]
    return q

anchor_quarters = quarterly_decomposition(anchor_history)
hedge_quarters = quarterly_decomposition(hedge_history)

quarter_compare = hedge_quarters.add_prefix("hedge_").join(anchor_quarters.add_prefix("anchor_"))
quarter_compare["delta_end_value"] = quarter_compare["hedge_end_value"] - quarter_compare["anchor_end_value"]
quarter_compare["delta_quarter_return_pct"] = quarter_compare["hedge_quarter_return_pct"] - quarter_compare["anchor_quarter_return_pct"]
quarter_compare["delta_quarter_dd_pct"] = quarter_compare["hedge_quarter_dd_pct"] - quarter_compare["anchor_quarter_dd_pct"]
quarter_compare.tail(12)

## Why Red-Regime Hedge Hurt in the Sweep

The refinement found that `g3=5%`, `g2=2.5%`, `red=0%` was the strict winner. Adding a red-regime hedge at 10% or 20% kept the same max drawdown but reduced final USD/SOL. This cell reproduces those three variants side by side.

In [ ]:
RED_FLOOR_VARIANTS = {
    "red_0pct": {4: 0.00, 3: 0.050, 2: 0.025, 1: 0.00, 0: 0.00},
    "red_10pct": {4: 0.00, 3: 0.050, 2: 0.025, 1: 0.10, 0: 0.10},
    "red_20pct": {4: 0.00, 3: 0.050, 2: 0.025, 1: 0.20, 0: 0.20},
}

red_variant_rows = []
red_variant_histories = {}
for name, floors in RED_FLOOR_VARIANTS.items():
    scenario = {
        "name": name,
        "enable_protective_short_hedge": True,
        "protective_short_symbol": "ETH",
        "protective_hedge_floors": floors,
    }
    result, _ = run_scenario(price_data, green_scores, scenario)
    hist = enrich_history(result.history, price_data)
    red_variant_histories[name] = hist
    red_variant_rows.append(metric_row(name, result, hist, FINAL_SOL_PRICE))

format_metrics(pd.DataFrame(red_variant_rows))

In [ ]:
svg_lines(
    {name: hist["portfolio_value"] for name, hist in red_variant_histories.items()},
    "Red-Floor Variants, Portfolio Value",
    y_label="USD",
)

In [ ]:
svg_lines(
    {name: hist["drawdown_pct"] for name, hist in red_variant_histories.items()},
    "Red-Floor Variants, Drawdown",
    y_label="Drawdown %",
)

## Custom Hedge Floor Test

Edit `CUSTOM_HEDGE_FLOORS` near the top, then run this cell. This is deliberately one scenario at a time so notebook use stays exploratory rather than turning into another broad optimizer.

In [ ]:
custom_scenario = {
    "name": "custom_hedge_floors",
    "enable_protective_short_hedge": True,
    "protective_short_symbol": "ETH",
    "protective_hedge_floors": CUSTOM_HEDGE_FLOORS,
}
custom_result, custom_config = run_scenario(price_data, green_scores, custom_scenario)
custom_history = enrich_history(custom_result.history, price_data)

custom_metrics = pd.DataFrame([
    metric_row("anchor_no_hedge", anchor_result, anchor_history, FINAL_SOL_PRICE),
    metric_row("current_best", hedge_result, hedge_history, FINAL_SOL_PRICE),
    metric_row("custom", custom_result, custom_history, FINAL_SOL_PRICE),
])
format_metrics(custom_metrics)

In [ ]:
svg_lines(
    {
        "Anchor": anchor_history["portfolio_value"],
        "Current best": hedge_history["portfolio_value"],
        "Custom": custom_history["portfolio_value"],
    },
    "Custom Scenario vs Anchor and Current Best",
    y_label="USD",
)

## Research Notes

Use this section after running the notebook to record observations before changing strategy code.

Current interpretation from the committed refinement:

- The model is still mainly a long SOL compounding strategy expressed in USD terms.
- The protective ETH short is useful only when it is light and early enough to affect the stress path without suppressing the rebound.
- The winning floor structure was `green4=0%`, `green3=5%`, `green2=2.5%`, `green1=0%`, `green0=0%`.
- Red-regime short floors were harmful in the tested grid because they added recovery drag without improving max drawdown.
- The next development stage should focus on path-aware hedge entry and exit rules, not simply increasing hedge size.